# Preparations

In [1]:
!pip -q install implicit rectools==0.4.2 lightfm nmslib optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 36.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 kB 23.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.6/409.6 kB 40.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.6/230.6 kB 29.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 10.1 MB/s eta 0:00:00


In [2]:
import typing as tp
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import implicit

from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking
from implicit.lmf import LogisticMatrixFactorization
from lightfm import LightFM
from rectools import Columns
from rectools.dataset import Dataset
from rectools.metrics import (
    MAP,
    NDCG,
    MeanInvUserFreq,
    Precision,
    Recall,
    Serendipity,
    calc_metrics,
)
from rectools.model_selection import TimeRangeSplitter, cross_validate
from rectools.models import ImplicitALSWrapperModel, LightFMWrapperModel, PopularModel
from rectools.tools import UserToItemAnnRecommender
from tqdm import tqdm

import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"  # For implicit ALS
import warnings

warnings.filterwarnings("ignore")

In [3]:
implicit.gpu.HAS_CUDA

True

In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# Loading data

In [5]:
DATA_PATH = Path("/content/drive/MyDrive/kion_train/")
users = pd.read_csv(DATA_PATH / "users.csv")
items = pd.read_csv(DATA_PATH / "items.csv")
interactions = pd.read_csv(DATA_PATH / "interactions.csv")

In [6]:
Columns.Datetime = "last_watch_dt"
interactions.drop(interactions[interactions[Columns.Datetime].str.len() != 10].index, inplace=True)
interactions[Columns.Datetime] = pd.to_datetime(interactions[Columns.Datetime], format="%Y-%m-%d")
max_date = interactions[Columns.Datetime].max()
interactions[Columns.Weight] = np.where(interactions["watched_pct"] > 10, 3, 1)

In [7]:
# разделим датасет на три части: на валидации будем подбирать гиперпараметры, на тесте финально сравнивать модели
train = interactions[interactions[Columns.Datetime] < max_date - pd.Timedelta(days=7)].copy()
test = interactions[interactions[Columns.Datetime] >= max_date - pd.Timedelta(days=7)].copy()

train.drop(train.query("total_dur < 300").index, inplace=True)

# отфильтруем холодных пользователей
cold_users = set(test[Columns.User]) - set(train[Columns.User])
test.drop(test[test[Columns.User].isin(cold_users)].index, inplace=True)

TEST_USERS = test[Columns.User].unique()

print(f"train: {train.shape}")
print(f"test: {test.shape}")

train: (3832711, 6)
test: (333026, 6)


## Preparing features

In [8]:
def get_user_features(users: pd.DataFrame, interactions: pd.DataFrame, features: tp.List[str]):
    users.fillna("Unknown", inplace=True)
    users = users.loc[users[Columns.User].isin(interactions[Columns.User])].copy()
    user_features_frames = []
    for feature in features:
        feature_frame = users.reindex(columns=[Columns.User, feature])
        feature_frame.columns = ["id", "value"]
        feature_frame["feature"] = feature
        user_features_frames.append(feature_frame)
    user_features = pd.concat(user_features_frames)
    return user_features

In [9]:
user_features = get_user_features(users, train, ["sex", "age", "income"])

In [10]:
def get_item_features(items: pd.DataFrame, interactions: pd.DataFrame):
    items = items.loc[items[Columns.Item].isin(interactions[Columns.Item])].copy()
    items["genre"] = items["genres"].str.lower().str.replace(", ", ",", regex=False).str.split(",")
    genre_feature = items[["item_id", "genre"]].explode("genre")
    genre_feature.columns = ["id", "value"]
    genre_feature["feature"] = "genre"
    content_feature = items.reindex(columns=[Columns.Item, "content_type"])
    content_feature.columns = ["id", "value"]
    content_feature["feature"] = "content_type"
    item_features = pd.concat((genre_feature, content_feature))
    return item_features

In [11]:
item_features = get_item_features(items, train)

## Constructing the dataset

In [12]:
dataset = Dataset.construct(
    interactions_df=train,
    user_features_df=user_features,
    cat_user_features=["sex", "age", "income"],
    item_features_df=item_features,
    cat_item_features=["genre", "content_type"],
)

# Hyperparam tuning

In [13]:
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.INFO)

In [14]:
K_RECOS = 10
RANDOM_STATE = 42
N_EPOCHS = 1  # Lightfm

In [15]:
def ALS_objective(trial, dataset, train, test):
    test_users = test[Columns.User].unique()
    metrics = {"MAP@10": MAP(k=10)}
    factors = trial.suggest_categorical("n_factors", [8, 16, 32])
    num_threads = trial.suggest_int("num_threads", 1, 3)
    fit_features_together = trial.suggest_categorical("fit_features_together", [True, False])

    model = ImplicitALSWrapperModel(
        model=AlternatingLeastSquares(
            factors=factors,
            random_state=RANDOM_STATE,
            num_threads=num_threads,
        ),
        fit_features_together=fit_features_together,
    )

    model.fit(dataset)
    recos = model.recommend(
        users=test_users,
        dataset=dataset,
        k=K_RECOS,
        filter_viewed=True,
    )
    metric_values = calc_metrics(metrics, recos, test, train)
    return metric_values["MAP@10"]

In [16]:
sampler = TPESampler(seed=1)
study = optuna.create_study(study_name="ALS", direction="maximize", sampler=sampler)
study.optimize(lambda trial: ALS_objective(trial, dataset, train, test), n_trials=20)

[I 2023-12-06 07:45:55,553] A new study created in memory with name: ALS
[I 2023-12-06 07:46:47,705] Trial 0 finished with value: 0.07523420163760208 and parameters: {'n_factors': 16, 'num_threads': 1, 'fit_features_together': True}. Best is trial 0 with value: 0.07523420163760208.


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

[I 2023-12-06 07:47:26,750] Trial 1 finished with value: 0.06376908131421508 and parameters: {'n_factors': 32, 'num_threads': 2, 'fit_features_together': False}. Best is trial 0 with value: 0.07523420163760208.


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

[I 2023-12-06 07:52:27,484] Trial 8 finished with value: 0.06937870315865291 and parameters: {'n_factors': 8, 'num_threads': 3, 'fit_features_together': False}. Best is trial 0 with value: 0.07523420163760208.


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

[I 2023-12-06 07:53:06,819] Trial 9 finished with value: 0.06377164689649542 and parameters: {'n_factors': 32, 'num_threads': 1, 'fit_features_together': False}. Best is trial 0 with value: 0.07523420163760208.
[I 2023-12-06 07:53:53,853] Trial 10 finished with value: 0.07226265700613856 and parameters: {'n_factors': 8, 'num_threads': 1, 'fit_features_together': True}. Best is trial 0 with value: 0.07523420163760208.
[I 2023-12-06 07:54:43,442] Trial 11 finished with value: 0.07458589723275587 and parameters: {'n_factors': 16, 'num_threads': 2, 'fit_features_together': True}. Best is trial 0 with value: 0.07523420163760208.
[I 2023-12-06 07:55:32,682] Trial 12 finished with value: 0.0749948062343709 and parameters: {'n_factors': 16, 'num_threads': 1, 'fit_features_together': True}. Best is trial 0 with value: 0.07523420163760208.
[I 2023-12-06 07:56:23,216] Trial 13 finished with value: 0.07498554718532449 and parameters: {'n_factors': 16, 'num_threads': 2, 'fit_features_together': Tru

In [17]:
def lightfm_objective(trial, dataset, train, test):
    test_users = test[Columns.User].unique()
    metrics = {"MAP@10": MAP(k=10)}
    no_components = trial.suggest_categorical("n_factors", [8, 16, 32, 64])
    loss = trial.suggest_categorical("loss", ["logistic", "bpr", "warp"])
    learning_rate = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    num_threads = trial.suggest_int("num_threads", 1, 3)
    user_alpha = trial.suggest_float("user_alpha", 0, 1)
    item_alpha = trial.suggest_float("item_alpha", 0, 1)

    model = LightFMWrapperModel(
        LightFM(
            no_components=no_components,
            loss=loss,
            random_state=RANDOM_STATE,
            learning_rate=learning_rate,
            user_alpha=user_alpha,
            item_alpha=item_alpha,
        ),
        epochs=N_EPOCHS,
        num_threads=num_threads,
    )

    model.fit(dataset)
    recos = model.recommend(
        users=test_users,
        dataset=dataset,
        k=K_RECOS,
        filter_viewed=True,
    )
    metric_values = calc_metrics(metrics, recos, test, train)
    return metric_values["MAP@10"]

In [18]:
sampler = TPESampler(seed=1)
study = optuna.create_study(study_name="lightFM", direction="maximize", sampler=sampler)
study.optimize(lambda trial: lightfm_objective(trial, dataset, train, test), n_trials=20)

[I 2023-12-06 08:01:15,389] A new study created in memory with name: lightFM
[I 2023-12-06 08:02:01,962] Trial 0 finished with value: 0.0053551460639551796 and parameters: {'n_factors': 16, 'loss': 'warp', 'lr': 0.0049104518184659674, 'num_threads': 2, 'user_alpha': 0.538816734003357, 'item_alpha': 0.4191945144032948}. Best is trial 0 with value: 0.0053551460639551796.
[I 2023-12-06 08:02:50,499] Trial 1 finished with value: 0.00019192744523331218 and parameters: {'n_factors': 32, 'loss': 'logistic', 'lr': 0.0019088591198098556, 'num_threads': 1, 'user_alpha': 0.8007445686755367, 'item_alpha': 0.9682615757193975}. Best is trial 0 with value: 0.0053551460639551796.
[I 2023-12-06 08:05:02,641] Trial 2 finished with value: 0.06494712932354058 and parameters: {'n_factors': 64, 'loss': 'warp', 'lr': 0.05705385668376793, 'num_threads': 1, 'user_alpha': 0.42110762500505217, 'item_alpha': 0.9578895301505019}. Best is trial 2 with value: 0.06494712932354058.
[I 2023-12-06 08:07:12,681] Trial 3 

# Cross-validation

## Models

Сравним лучшие модели на кросс-валидации

In [19]:
models = {
    "popular": PopularModel(),
    "ALS": ImplicitALSWrapperModel(
        model=AlternatingLeastSquares(
            factors=32,
            random_state=RANDOM_STATE,
            num_threads=2,
        ),
        fit_features_together=True,
    ),
    "LightFM": LightFMWrapperModel(
        LightFM(
            no_components=8,
            loss="warp",
            random_state=RANDOM_STATE,
            learning_rate=0.05,
            user_alpha=0.3,
            item_alpha=0.2,
        ),
        epochs=N_EPOCHS,
        num_threads=2,
    ),
}

## Metrics

In [20]:
metrics_name = {
    "precision": Precision,
    "recall": Recall,
    "MAP": MAP,
    "NDCG": NDCG,
    "novelty": MeanInvUserFreq,
    "serendipity": Serendipity,
}

metrics = {}
for metric_name, metric in metrics_name.items():
    for k in [1, 5, 10]:
        metrics[f"{metric_name}@{k}"] = metric(k=k)

In [21]:
metrics

{'precision@1': Precision(k=1),
 'precision@5': Precision(k=5),
 'precision@10': Precision(k=10),
 'recall@1': Recall(k=1),
 'recall@5': Recall(k=5),
 'recall@10': Recall(k=10),
 'MAP@1': MAP(k=1, divide_by_k=False),
 'MAP@5': MAP(k=5, divide_by_k=False),
 'MAP@10': MAP(k=10, divide_by_k=False),
 'NDCG@1': NDCG(k=1, log_base=2),
 'NDCG@5': NDCG(k=5, log_base=2),
 'NDCG@10': NDCG(k=10, log_base=2),
 'novelty@1': MeanInvUserFreq(k=1),
 'novelty@5': MeanInvUserFreq(k=5),
 'novelty@10': MeanInvUserFreq(k=10),
 'serendipity@1': Serendipity(k=1),
 'serendipity@5': Serendipity(k=5),
 'serendipity@10': Serendipity(k=10)}

## Splitter

In [22]:
TEST_SIZE = "7D"
N_SPLITS = 5

In [23]:
splitter = TimeRangeSplitter(
    test_size=TEST_SIZE,
    n_splits=N_SPLITS,
    filter_already_seen=True,
    filter_cold_items=True,
    filter_cold_users=True,
)

In [24]:
splitter.get_test_fold_borders(dataset.interactions)

[(Timestamp('2021-07-11 00:00:00', freq='7D'),
  Timestamp('2021-07-18 00:00:00', freq='7D')),
 (Timestamp('2021-07-18 00:00:00', freq='7D'),
  Timestamp('2021-07-25 00:00:00', freq='7D')),
 (Timestamp('2021-07-25 00:00:00', freq='7D'),
  Timestamp('2021-08-01 00:00:00', freq='7D')),
 (Timestamp('2021-08-01 00:00:00', freq='7D'),
  Timestamp('2021-08-08 00:00:00', freq='7D')),
 (Timestamp('2021-08-08 00:00:00', freq='7D'),
  Timestamp('2021-08-15 00:00:00', freq='7D'))]

## Cross-val

In [25]:
results = cross_validate(dataset, splitter, metrics, models, k=10, filter_viewed=True)

In [26]:
df_quality = (
    pd.DataFrame.from_dict(results["metrics"]).groupby("model").mean().drop("i_split", axis=1).T
)
df_quality.style.highlight_max(color="lightgreen", axis=1)

model,ALS,LightFM,popular
precision@1,0.093644,0.064398,0.081874
recall@1,0.057804,0.039540,0.050963
precision@5,0.053747,0.041753,0.056047
recall@5,0.154709,0.119647,0.161825
precision@10,0.034727,0.027002,0.036284
recall@10,0.193501,0.153204,0.203790
NDCG@1,0.093644,0.064398,0.081874
NDCG@5,0.062128,0.046038,0.061691
NDCG@10,0.045924,0.034418,0.045989
MAP@1,0.057804,0.039540,0.050963


По большиству метрик **лучшей оказалась ALS**.

# Preparing model for the service

## Training on the whole dataset

Лучшей моделью оказалась ALS. Но нужен GPU.   
нет  GPU в сервисе, возьмем LightFM, которая не намного уступает ALS.  
Обучим на всем датасете.

In [27]:
user_features = get_user_features(users, interactions, ["sex", "age", "income"])
item_features = get_item_features(items, interactions)

In [28]:
dataset = Dataset.construct(
    interactions_df=interactions,
    user_features_df=user_features,
    cat_user_features=["sex", "age", "income"],
    item_features_df=item_features,
    cat_item_features=["genre", "content_type"],
)

In [29]:
model = LightFMWrapperModel(
    LightFM(
        no_components=8,
        loss="warp",
        random_state=RANDOM_STATE,
        learning_rate=0.05,
        user_alpha=0.3,
        item_alpha=0.2,
    ),
    epochs=N_EPOCHS,
    num_threads=2,
)

In [30]:
model.fit(dataset)

# Offline recommendations

Поэтому посчитаем оффлайн рекомендации и сохраним их на диск, чтобы потом использовать в сервисе.

In [31]:
ALL_USERS = interactions[Columns.User].unique()

In [32]:
all_recos = model.recommend(
    users=ALL_USERS,
    dataset=dataset,
    k=10,
    filter_viewed=True,
)[[Columns.User, Columns.Item]]

In [34]:
RECOS_PATH = "/content/drive/MyDrive/kion_train/LightFM_warp_8.csv"
all_recos.to_csv(RECOS_PATH)

# Adding ANN

Чтобы рекомендации выдавались быстрее, можно использовать приближенный поиск соседей из `rectools`.

In [35]:
user_vectors, item_vectors = model.get_vectors(dataset)
ann_lightfm = UserToItemAnnRecommender(
    user_vectors=user_vectors,
    item_vectors=item_vectors,
    user_id_map=dataset.user_id_map,
    item_id_map=dataset.item_id_map,
)
ann_lightfm.fit()

In [36]:
# Поробуем получить рекомендации
ann_lightfm.get_item_list_for_user(962205, top_n=10).tolist()

[12138, 12089, 6577, 16080, 15550, 1679, 2981, 16026, 7823, 734]

In [37]:
import pickle

MODEL_PATH = "/content/drive/MyDrive/kion_train/ANN_LightFM_warp_8.pkl"
pickle.dump(ann_lightfm, open(MODEL_PATH, "wb"))

In [38]:
# Проверим, что все работает
loaded_ann_lightfm = pickle.load(open(MODEL_PATH, "rb"))
ann_lightfm.get_item_list_for_user(962205, top_n=10).tolist()

[12138, 12089, 6577, 16080, 15550, 1679, 2981, 16026, 7823, 734]